# Mahagauri: has the server really recovered?

327 real server incidents, minute by minute from onset to an hour after. Declare the incident over once the model says 'recovered' for several minutes running. The dangerous error is declaring it over while it is still going.

Mahagauri is one of the nine models of **Sati**, the investigation agent of [Pashupatastra](https://github.com/preetraval45/PASHUPATASTRA), each named for a form of Navadurga. This notebook reproduces its core result on the public dataset, with the split that keeps the test honest.

In [ ]:
import glob, os
def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"{pattern} not found under /kaggle/input"
    return hits[0]
import pandas as pd, numpy as np
d = pd.read_parquet(find("recovery.parquet"))
feats = [c for c in d.columns if c not in ("incident", "group", "recovered")]
print(d["incident"].nunique(), "incidents")

In [ ]:
import lightgbm as lgb
def outcome(recovered, ok, hold):
    end = int(recovered.argmax()) if recovered.any() else len(recovered)
    run = 0
    for i, v in enumerate(ok):
        run = run + 1 if v else 0
        if run >= hold:
            return "premature" if i < end else "confirmed"
    return "unconfirmed"
rows = []
for held in sorted(d["group"].unique()):
    tr, te = d[d["group"] != held], d[d["group"] == held].copy()
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1).fit(tr[feats], tr["recovered"])
    te["p"] = m.predict_proba(te[feats])[:, 1]
    for t, hold in ((0.5, 3), (0.9, 10)):
        res = pd.Series([outcome(g["recovered"].values, g["p"].values >= t, hold) for _, g in te.groupby("incident")])
        rows.append({"held out": held, "threshold": t, "hold (min)": hold, **res.value_counts(normalize=True).round(3).to_dict()})
pd.DataFrame(rows).fillna(0)

**Reading it:** a lenient rule confirms more recoveries but declares many too early; a strict one is safe and rarely says anything. Mahagauri, tuned on out-of-fold scores, keeps premature all-clears at 0.021 on unseen groups but confirms only 0.092: making a verifier both safe and useful is open.